# Comparing models across historical windows

The previous notebooks compared models mainly on 2011–2018. Here we ask whether the average/trend blend and XGBoost also perform well in other periods, using the same reference baselines and starter Random Forest. This is an exploratory analysis; no submission file is generated.

## How the comparison works

Each fold simulates standing at a historical cutoff and predicting the next eight years. For example, at the end of 2002 we train on data through 2002, then forecast 2003–2010. XGBoost and the Random Forest use their own earlier predictions when forecasting later years.

For each fold, the blend's weight is selected separately for each metric using the **preceding** eight-year period. That selection ends at the fold cutoff. It never uses the outcomes of the period being scored. Every fold ends by 2018.

Earlier experiments have already inspected 2011–2018, so this is now an exploratory comparison, not a completely untouched final test. The folds also reuse historical data across windows; their results are not statistically independent.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'code' / 'common.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Open this notebook inside the ProtoStar-Challenge repository.')

sys.path.insert(0, str(PROJECT_ROOT / 'code'))
from common import CUTOFF_YEAR, METRICS, load_valid_annual, thirty_year_mean, wide
from starter_model import baselines, forecast as forecast_rf

annual = load_valid_annual()  # Includes the repository's post-2018 data guard.
cities = pd.read_csv(PROJECT_ROOT / 'data' / 'city_info.csv', dtype={'city_id': str}).set_index('city_id')
print(f'Loaded {len(annual):,} guarded city-years for {annual.city_id.nunique()} cities.')

## Reuse the existing XGBoost experiment

Notebook 04 contains the fixed features, hyperparameters, and recursive forecast function. The following cell loads **only its two definition cells**; it does not execute its training, evaluation, plots, or final insights. This preserves the exact experiment we are comparing. Keep notebook 04 beside this notebook.

Its training and prediction feature functions handle sparse rolling windows slightly differently; that existing limitation means the comparison is exploratory and should be corrected and revalidated before any model is promoted.

In [ ]:
import ast

xgb_path = PROJECT_ROOT / 'notebooks' / 'Achintya' / '04_xgboost_experiment.ipynb'
xgb_notebook = json.loads(xgb_path.read_text(encoding='utf-8'))
for marker in ('FEATURE_COLUMNS = [', 'XGB_PARAMS = {'):
    matches = [cell for cell in xgb_notebook['cells']
               if cell['cell_type'] == 'code' and ''.join(cell['source']).lstrip().startswith(marker)]
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one definition cell starting with {marker!r} in {xgb_path.name}.')
    tree = ast.parse(''.join(matches[0]['source']))
    allowed_names = {'FEATURE_COLUMNS', 'XGB_PARAMS'}
    definitions = [node for node in tree.body if isinstance(node, ast.FunctionDef) or
                   (isinstance(node, ast.Assign) and len(node.targets) == 1 and
                    isinstance(node.targets[0], ast.Name) and node.targets[0].id in allowed_names)]
    exec(compile(ast.Module(body=definitions, type_ignores=[]), str(xgb_path), 'exec'))
print('Loaded the fixed XGBoost definitions from notebook 04.')

## Define the folds

The final window matches the existing 2011–2018 backtest. Earlier windows provide additional evidence about stability; the training histories expand as the cutoff advances.

In [ ]:
FOLD_CUTOFFS = [1994, 2002, 2010]
HORIZON = 8
WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]

fold_plan = pd.DataFrame([
    {'train_through': cutoff,
     'weight_selection_train_through': cutoff - HORIZON,
     'weight_selection_period': f'{cutoff - HORIZON + 1}–{cutoff}',
     'evaluation_period': f'{cutoff + 1}–{cutoff + HORIZON}'}
    for cutoff in FOLD_CUTOFFS
])
assert all(cutoff + HORIZON <= CUTOFF_YEAR for cutoff in FOLD_CUTOFFS)
fold_plan

## Select the blend weight using earlier data

Weight `0` means the 30-year average; weight `1` means the linear trend. We compare all candidates on the same eligible city-year rows, requiring a valid outcome and a 30-year baseline with at least 10 valid training years. If any candidate is missing a prediction for an eligible row, the notebook stops instead of silently changing the scoring population.

In [ ]:
def aligned_errors(actual: pd.DataFrame, predictions: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Return absolute errors for the same eligible city-years for every model."""
    actual_long = actual.stack(future_stack=True).rename('actual').dropna()
    if actual_long.empty:
        raise ValueError('No eligible observations for this period.')
    errors = pd.DataFrame(index=actual_long.index)
    for name, prediction in predictions.items():
        pred = prediction.reindex(index=actual.index, columns=actual.columns).stack(future_stack=True).reindex(actual_long.index)
        if not np.isfinite(pred.to_numpy(dtype=float)).all():
            raise ValueError(f'{name} has missing or non-finite predictions on eligible rows.')
        errors[name] = (actual_long - pred).abs()
    return errors

def select_weight(table: pd.DataFrame, cutoff: int) -> tuple[float, pd.DataFrame]:
    selection_cutoff = cutoff - HORIZON
    years = list(range(selection_cutoff + 1, cutoff + 1))
    references = baselines(table, selection_cutoff, years)
    average, trend = references['30_year_average'], references['linear_trend']
    eligible = thirty_year_mean(table, selection_cutoff).notna()
    actual = table[years].where(eligible, axis=0)
    candidates = {str(weight): (1 - weight) * average + weight * trend for weight in WEIGHTS}
    errors = aligned_errors(actual, candidates)
    scores = errors.mean()
    # The ordered grid resolves exact ties in favour of the lower trend weight.
    best = min(WEIGHTS, key=lambda weight: scores[str(weight)])
    return best, scores.rename('mae').rename_axis('trend_weight').reset_index()

## Run the historical comparisons

This fits one XGBoost and one Random Forest per metric per fold on CPU. It can take a few minutes. Each model is scored on exactly the same eligible rows; the `n_scored` column records the sample size.

In [ ]:
rows = []
weight_rows = []
for cutoff in FOLD_CUTOFFS:
    years = list(range(cutoff + 1, cutoff + HORIZON + 1))
    for metric in METRICS:
        print(f'Train through {cutoff}; evaluating {metric}...', flush=True)
        table = wide(annual, metric)
        weight, selection_scores = select_weight(table, cutoff)
        weight_rows.append({'train_through': cutoff, 'metric': metric, 'trend_weight': weight})
        refs = baselines(table, cutoff, years)
        blend = (1 - weight) * refs['30_year_average'] + weight * refs['linear_trend']
        xgb_prediction, _ = forecast_xgboost(table, cutoff, years)
        predictions = {**refs, 'starter_rf_lags': forecast_rf(table, cutoff, years),
                       'selected_average_trend_blend': blend, 'xgboost_engineered': xgb_prediction}
        eligible = thirty_year_mean(table, cutoff).notna()
        actual = table[years].where(eligible, axis=0)
        errors = aligned_errors(actual, predictions)
        for model in predictions:
            rows.append({'train_through': cutoff, 'evaluation': f'{years[0]}–{years[-1]}',
                         'metric': metric, 'model': model,
                         'mae': errors[model].mean(), 'n_scored': len(errors)})

results = pd.DataFrame(rows)
selected_weights = pd.DataFrame(weight_rows)
reference = results.loc[results.model.eq('30_year_average'), ['train_through', 'metric', 'mae']].rename(columns={'mae': 'reference_mae'})
results = results.merge(reference, on=['train_through', 'metric'], validate='many_to_one')
results['skill'] = 1 - results['mae'] / results['reference_mae']
print('Finished all historical folds.')
selected_weights.pivot(index='metric', columns='train_through', values='trend_weight')

## Compare results

MAE is in each metric's own units, so compare model errors **within** a metric. The challenge-style score averages each metric's improvement relative to its 30-year-average baseline and clips negative overall scores to zero. These are historical estimates, not official leaderboard scores.

In [ ]:
mae_comparison = results.pivot_table(index=['evaluation', 'metric'], columns='model', values='mae')
display(mae_comparison.round(3))
fold_scores = (100 * results.groupby(['evaluation', 'model'])['skill'].mean()).clip(lower=0).rename('score').reset_index()
display(fold_scores.pivot(index='model', columns='evaluation', values='score').round(2))
display(results.groupby(['evaluation', 'metric'])['n_scored'].first().unstack().astype(int))

In [ ]:
focus = results[results.model.isin(['30_year_average', 'linear_trend', 'selected_average_trend_blend', 'xgboost_engineered'])]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for axis, metric in zip(axes.flat, METRICS):
    subset = focus[focus.metric.eq(metric)].pivot(index='evaluation', columns='model', values='mae')
    subset.plot(marker='o', ax=axis, legend=False)
    axis.set_title(metric)
    axis.set_ylabel('MAE (metric-specific units)')
    axis.set_xlabel('Evaluation period')
    axis.tick_params(axis='x', rotation=20)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.04))
plt.tight_layout()

## Key insights and next decision

- The average/trend blend scored **3.95, 5.24, and 6.47** across the three successive evaluation windows. It outperformed the 30-year-average reference overall in all three periods. Its weight was selected separately on earlier data in each fold.
- XGBoost scored **0, 0, and 5.81** across the same windows. Its CDD MAE was **163.70, 223.37, and 169.19**, respectively. It beat the 30-year CDD average only in 2011–2018; the earlier periods do not support treating that one win as stable.
- The blend's precipitation improvement is not consistent: its MAE was **5.68** versus **5.59** for the 30-year average in 1995–2002, and equal to it in 2003–2010.
- The number of scored city-years grows across folds, so differences between periods reflect different populations as well as changing climate conditions. These are eight-year historical windows, while the challenge forecast spans seven years.
- Notebook 04 handles sparse rolling windows differently during training and prediction. Correct that inconsistency and re-evaluate XGBoost before making a submission decision. These folds have been inspected during development and are not an independent final test.

The blend is the more consistent candidate so far. The team should review the code, investigate the XGBoost feature issue, and only then decide what to promote into the reproducible `code/` pipeline.